# Arabic Media Bias Detection — ARBERT Fine-tuning

**Runtime**: Google Colab (T4 GPU recommended)  
**Model**: UBC-NLP/ARBERTv2  
**Task**: Binary classification — Non-biased (0) vs Biased (1)

---

### Quick-start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells in order (`Runtime → Run all`)
3. Choose the experiment in **Cell 3**
4. Outputs saved to `outputs/<experiment_name>/`

No Google Drive required — everything runs in the Colab instance.

## 1 — Environment setup

In [ ]:
# Install dependencies (takes ~60 s on Colab)
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate>=0.30.0 evaluate>=0.4.0 \
                scikit-learn>=1.3.0 pandas>=2.0.0 seaborn>=0.12.0 pyyaml>=6.0

In [ ]:
import os

REPO_URL  = "https://github.com/YOUR_USERNAME/arabic-bias-detection.git"  # ← update this
REPO_DIR  = "arabic-bias-detection"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found — training will be slow. Set Runtime → T4 GPU.")

## 2 — Choose experiment

In [ ]:
# ── Choose one ─────────────────────────────────────────────────────────────
# EXPERIMENT = "configs/baseline_gpt_only.yaml"       # Experiment A: GPT-only
EXPERIMENT  = "configs/main_gpt_claude_mixed.yaml"    # Experiment B: GPT + Claude
# ───────────────────────────────────────────────────────────────────────────

print(f"Selected config: {EXPERIMENT}")

## 3 — Prepare data

In [ ]:
!python src/prepare_data.py --config {EXPERIMENT}

In [ ]:
import json, pandas as pd
from src.utils import load_config

cfg = load_config(EXPERIMENT)
exp = cfg["experiment_name"]

train_df = pd.read_csv(cfg["data"]["train_path"])
test_df  = pd.read_csv(cfg["data"]["test_path"])

with open(cfg["data"]["summary_path"]) as f:
    summary = json.load(f)

print(f"Experiment : {exp}")
print(f"Train rows : {len(train_df):,}")
print(f"Test  rows : {len(test_df):,}")
print(f"\nLabel distribution (train):")
print(train_df["label"].value_counts().rename({0:'Non-biased', 1:'Biased'}))

print(f"\nSample rows:")
train_df[["arabic_text","label","source"]].head(3)

## 4 — Train

In [ ]:
!python src/train.py --config {EXPERIMENT}

## 5 — Evaluate

In [ ]:
!python src/evaluate.py --config {EXPERIMENT}

In [ ]:
import json
from src.utils import load_config
from pathlib import Path

cfg = load_config(EXPERIMENT)
metrics_path = Path(cfg["output_dir"]) / "metrics.json"

with open(metrics_path) as f:
    m = json.load(f)

print(f"\n{'='*40}")
print(f"  {m['experiment']}")
print(f"{'='*40}")
print(f"  Accuracy        : {m['accuracy']:.4f}")
print(f"  F1 Macro        : {m['f1_macro']:.4f}")
print(f"  F1 Weighted     : {m['f1_weighted']:.4f}")
print(f"  Precision Macro : {m['precision_macro']:.4f}")
print(f"  Recall Macro    : {m['recall_macro']:.4f}")
print(f"  F1 Non-biased   : {m['f1_non_biased']:.4f}")
print(f"  F1 Biased       : {m['f1_biased']:.4f}")

In [ ]:
from IPython.display import Image, display
from pathlib import Path
from src.utils import load_config

cfg = load_config(EXPERIMENT)
cm_path = Path(cfg["output_dir"]) / "confusion_matrix.png"
display(Image(str(cm_path)))

## 6 — Inference on new sentences

In [ ]:
# Classify a few example sentences
test_sentences = [
    "أعلنت الحكومة عن خطة اقتصادية جديدة لتحسين مستوى المعيشة.",   # neutral
    "يسعى الإعلام المتحيز دائمًا إلى تشويه الحقائق لخدمة أجندات خاصة.",  # biased
    "التقرير يستعرض الأرقام الرسمية دون تعليق.",                     # neutral
]

for sent in test_sentences:
    !echo "{sent}" | python src/predict.py --config {EXPERIMENT}

## 7 — Download outputs

In [ ]:
from google.colab import files
from pathlib import Path
from src.utils import load_config

cfg = load_config(EXPERIMENT)
out = Path(cfg["output_dir"])

for fname in ["metrics.json", "classification_report.txt",
              "confusion_matrix.png", "predictions.csv"]:
    p = out / fname
    if p.exists():
        files.download(str(p))
        print(f"⬇️  {fname}")
    else:
        print(f"⚠️  Not found: {fname}")

---
## Notes

| Setting | Value |
|---------|-------|
| Default model | `UBC-NLP/ARBERTv2` |
| Switch model | Edit `model.name` in the YAML config |
| Alternative | `aubmindlab/bert-base-arabertv2` |
| Epochs | 5 (edit `training.epochs` in YAML) |
| Best checkpoint | Saved by F1-macro |
| Leakage guard | Group-aware split on `sentence_id` |

**Adding a third experiment (Gemini source):**  
Duplicate a config YAML, set `sources_to_include: [gpt, claude, gemini]`, run prepare + train + evaluate.